# 01 - project setup and verification (Beta-Lactam ML v2, sequence-only benchmark)

Run this notebook top to bottom on a fresh machine. It:
1. locates the project root and puts `paths.py` on the import path,
2. records pinned library versions,
3. detects and prints the compute device (GPU vs CPU) downstream code will use,
4. confirms the directory tree and the Firnberg raw data file exist,
5. ends with a printed PASS/FAIL checklist, one line per requirement.

A notebook that runs without error isn't the same thing as one that confirms every requirement actually holds. The checklist at the bottom is the point of this notebook.

Naming discipline (project rule 1): classifiers are always named individually, Logistic Regression, Random Forest, XGBoost, SVM, never "baselines." Splits are always named in full: random split, modulo split, contiguous (region-holdout) split, never "our splits."


## 1. Locate project root and import paths

In [1]:
import sys
from pathlib import Path

# Walk up from the notebook to find the .projectroot marker, then import paths.py.
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.projectroot').exists())
sys.path.insert(0, str(root))
from paths import BASE_DIR, RAW, INTERIM, PROCESSED, MODELS, FIGURES, TABLES

print('project root :', BASE_DIR)
for name, d in [('data/raw', RAW), ('data/interim', INTERIM), ('data/processed', PROCESSED),
                ('models', MODELS), ('results/figures', FIGURES), ('results/tables', TABLES)]:
    print(f'  {name:18} -> {d}')


project root : /Users/kdave2/Beta-Lactamase Mutagenesis/Beta Lactam ML v2
  data/raw           -> /Users/kdave2/Beta-Lactamase Mutagenesis/Beta Lactam ML v2/data/raw
  data/interim       -> /Users/kdave2/Beta-Lactamase Mutagenesis/Beta Lactam ML v2/data/interim
  data/processed     -> /Users/kdave2/Beta-Lactamase Mutagenesis/Beta Lactam ML v2/data/processed
  models             -> /Users/kdave2/Beta-Lactamase Mutagenesis/Beta Lactam ML v2/models
  results/figures    -> /Users/kdave2/Beta-Lactamase Mutagenesis/Beta Lactam ML v2/results/figures
  results/tables     -> /Users/kdave2/Beta-Lactamase Mutagenesis/Beta Lactam ML v2/results/tables


## 2. Library versions (pinned)

In [2]:
import importlib.metadata as im, platform

REQUIRED = {
    'scikit-learn': '1.9.0', 'xgboost': '3.2.0', 'pandas': '3.0.3', 'numpy': '2.4.6',
    'scipy': '1.17.1', 'pyarrow': '25.0.0', 'matplotlib': '3.11.0', 'seaborn': '0.13.2',
    'joblib': '1.5.3', 'statsmodels': '0.14.6',
}
print('python       :', platform.python_version())
installed = {}
for pkg, want in REQUIRED.items():
    try:
        got = im.version(pkg); installed[pkg] = got
        flag = 'ok' if got == want else f'DIFFERENT (pinned {want})'
        print(f'  {pkg:14}: {got:10} {flag}')
    except im.PackageNotFoundError:
        installed[pkg] = None
        print(f'  {pkg:14}: MISSING')


python       : 3.11.15
  scikit-learn  : 1.9.0      ok
  xgboost       : 3.2.0      ok
  pandas        : 3.0.3      ok
  numpy         : 2.4.6      ok
  scipy         : 1.17.1     ok
  pyarrow       : 25.0.0     ok
  matplotlib    : 3.11.0     ok
  seaborn       : 0.13.2     ok
  joblib        : 1.5.3      ok
  statsmodels   : 0.14.6     ok


## 3. Device detection

Prints whether a GPU is available and which device downstream code will use. This benchmark's traditional-ML stage (Logistic Regression, Random Forest, XGBoost, SVM) is CPU-only. The PLM-scoring phase (ESM2, ESM-1v, ESM C) runs in a separate env (`esm`, fair-esm + torch), so torch is intentionally absent here.

In [3]:
import os

# torch is not part of the betalactam-v2 ML env; detect gracefully.
try:
    import torch
    cuda = torch.cuda.is_available()
    mps  = getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available()
    device = 'cuda' if cuda else ('mps' if mps else 'cpu')
    print(f'torch        : {torch.__version__}')
    print(f'CUDA available: {cuda}')
    print(f'MPS available : {mps}')
    print(f'-> downstream torch device: {device}')
except ModuleNotFoundError:
    print('torch        : not installed in betalactam-v2 (expected).')
    print('CUDA available: False (torch absent; PLM phase uses the `esm` env)')
    print('-> traditional-ML stage runs on: CPU')

print(f'CPU cores     : {os.cpu_count()}')
try:
    import psutil
    vm = psutil.virtual_memory()
    print(f'RAM total     : {vm.total/1e9:.1f} GB')
    print(f'RAM available : {vm.available/1e9:.1f} GB')
except ModuleNotFoundError:
    print('RAM           : (install psutil to report memory)')


torch        : not installed in betalactam-v2 (expected).
CUDA available: False (torch absent; PLM phase uses the `esm` env)
-> traditional-ML stage runs on: CPU
CPU cores     : 8
RAM total     : 17.2 GB
RAM available : 4.5 GB


## 4. Data check: Firnberg (2014) TEM-1 DMS

The only carried-over dataset is the Firnberg (2014) TEM-1 deep mutational scanning file. Drop `BLAT_ECOLX_Firnberg_2014.csv` into `data/raw/` before running the pipeline. Standard ProteinGym schema: `mutant, mutated_sequence, DMS_score, DMS_score_bin`.

In [4]:
import pandas as pd, glob

firnberg = sorted(glob.glob(str(RAW / '*Firnberg*.csv')) + glob.glob(str(RAW / '*firnberg*.csv')))
if firnberg:
    fpath = Path(firnberg[0])
    df = pd.read_csv(fpath)
    print(f'found        : {fpath.name}')
    print(f'rows         : {len(df)}')
    print(f'columns      : {list(df.columns)}')
    print(f'size on disk : {fpath.stat().st_size/1e6:.2f} MB')
    if 'DMS_score_bin' in df.columns:
        print(f'label balance: {df.DMS_score_bin.mean():.3f} positive')
else:
    df = None
    print('NOT FOUND    : no *Firnberg*.csv in', RAW)
    print('  -> place BLAT_ECOLX_Firnberg_2014.csv in data/raw/ and re-run this cell.')


found        : BLAT_ECOLX_Firnberg_2014.csv
rows         : 4783
columns      : ['mutant', 'mutated_sequence', 'DMS_score', 'DMS_score_bin']
size on disk : 1.44 MB
label balance: 0.501 positive


## 5. Setup verification checklist (pass/fail)

In [5]:
checks = []

# 5a: every dependency imports
import importlib
for mod in ['sklearn', 'xgboost', 'pandas', 'numpy', 'scipy', 'pyarrow',
            'matplotlib', 'seaborn', 'joblib', 'statsmodels']:
    try:
        importlib.import_module(mod); checks.append((f'import {mod}', True, ''))
    except Exception as e:
        checks.append((f'import {mod}', False, str(e)[:50]))

# 5b: directory tree exists
for name, d in [('data/raw', RAW), ('data/interim', INTERIM), ('data/processed', PROCESSED),
                ('models', MODELS), ('results/figures', FIGURES), ('results/tables', TABLES)]:
    checks.append((f'dir {name}', Path(d).is_dir(), '' if Path(d).is_dir() else 'missing'))

# 5c: Firnberg raw data present and well-formed
ok_data = firnberg is not None and len(firnberg) > 0
checks.append(('Firnberg CSV in data/raw', ok_data,
               '' if ok_data else 'drop BLAT_ECOLX_Firnberg_2014.csv into data/raw'))
if ok_data:
    need = {'mutant', 'DMS_score'}
    have = need.issubset(set(df.columns))
    checks.append(('Firnberg schema (mutant, DMS_score)', have,
                   '' if have else f'missing {need - set(df.columns)}'))

# 5d: device explicitly known
checks.append(('device detected (see section 3)', True, ''))

print('=' * 64)
print('SETUP VERIFICATION CHECKLIST')
print('=' * 64)
n_pass = 0
for label, ok, note in checks:
    tag = 'PASS' if ok else 'FAIL'
    n_pass += int(ok)
    print(f'[{tag}] {label:42} {note}')
print('=' * 64)
print(f'{n_pass}/{len(checks)} checks passed'
      + ('  - setup OK' if n_pass == len(checks) else '  - RESOLVE FAILs above'))


SETUP VERIFICATION CHECKLIST
[PASS] import sklearn                             
[PASS] import xgboost                             
[PASS] import pandas                              
[PASS] import numpy                               
[PASS] import scipy                               
[PASS] import pyarrow                             
[PASS] import matplotlib                          
[PASS] import seaborn                             
[PASS] import joblib                              
[PASS] import statsmodels                         
[PASS] dir data/raw                               
[PASS] dir data/interim                           
[PASS] dir data/processed                         
[PASS] dir models                                 
[PASS] dir results/figures                        
[PASS] dir results/tables                         
[PASS] Firnberg CSV in data/raw                   
[PASS] Firnberg schema (mutant, DMS_score)        
[PASS] device detected (see section 3)            
19